In [1]:
from langgraph.graph import StateGraph,MessagesState, START, END
from pydantic import BaseModel

In [2]:
from typing import Literal, Optional


class Snowflake(BaseModel):
    SNOWFLAKE_ACCOUNT: str
    SNOWFLAKE_USER: str
    SNOWFLAKE_AUTH_METHOD: Literal["password", "key_pair", "externalbrowser"] = "password"
    SNOWFLAKE_PASSWORD: Optional[str] = None
    SNOWFLAKE_PRIVATE_KEY_PATH: Optional[str] = None
    SNOWFLAKE_PRIVATE_KEY_PASSPHRASE: Optional[str] = None
    SNOWFLAKE_ROLE: Optional[str] = None
    SNOWFLAKE_WAREHOUSE: Optional[str] = None
    SNOWFLAKE_DATABASE: Optional[str] = None
    SNOWFLAKE_SCHEMA: Optional[str] = None
    SNOWFLAKE_QUERY_TAG: str = "snowflake-agent"
    SNOWFLAKE_LOGIN_TIMEOUT: int = 60
    SNOWFLAKE_NETWORK_TIMEOUT: int = 300
    SNOWFLAKE_STATEMENT_TIMEOUT: int = 120  # Snowflake cancels agent queries running longer than this

In [3]:
import os

import snowflake.connector
from cryptography.hazmat.primitives import serialization
from dotenv import find_dotenv, load_dotenv
from langchain_core.tools import tool

load_dotenv(find_dotenv(usecwd=True))


def get_snowflake_config() -> Snowflake:
    fields = Snowflake.model_fields
    return Snowflake(**{k: os.environ[k] for k in fields if os.environ.get(k)})


def get_snowflake_connection(cfg: Snowflake) -> snowflake.connector.SnowflakeConnection:
    params = {
        "account": cfg.SNOWFLAKE_ACCOUNT,
        "user": cfg.SNOWFLAKE_USER,
        "role": cfg.SNOWFLAKE_ROLE,
        "warehouse": cfg.SNOWFLAKE_WAREHOUSE,
        "database": cfg.SNOWFLAKE_DATABASE,
        "schema": cfg.SNOWFLAKE_SCHEMA,
        "login_timeout": cfg.SNOWFLAKE_LOGIN_TIMEOUT,
        "network_timeout": cfg.SNOWFLAKE_NETWORK_TIMEOUT,
        "session_parameters": {
            "QUERY_TAG": cfg.SNOWFLAKE_QUERY_TAG,
            # the warehouse default is 2 days; cap runaway agent queries so they stop burning credits
            "STATEMENT_TIMEOUT_IN_SECONDS": cfg.SNOWFLAKE_STATEMENT_TIMEOUT,
        },
    }
    if cfg.SNOWFLAKE_AUTH_METHOD == "password":
        params["password"] = cfg.SNOWFLAKE_PASSWORD
    elif cfg.SNOWFLAKE_AUTH_METHOD == "key_pair":
        with open(os.path.expanduser(cfg.SNOWFLAKE_PRIVATE_KEY_PATH), "rb") as f:
            passphrase = cfg.SNOWFLAKE_PRIVATE_KEY_PASSPHRASE
            key = serialization.load_pem_private_key(
                f.read(), password=passphrase.encode() if passphrase else None
            )
        params["private_key"] = key.private_bytes(
            encoding=serialization.Encoding.DER,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption(),
        )
    else:
        params["authenticator"] = "externalbrowser"
    return snowflake.connector.connect(**{k: v for k, v in params.items() if v is not None})

In [ ]:
@tool
def retrieval() -> dict:
    """List the databases available in Snowflake, with the schemas and tables/views in each.
    Returns {database: {schema: [table, ...]}}."""
    conn = get_snowflake_connection(get_snowflake_config())
    try:
        cur = conn.cursor(snowflake.connector.DictCursor)

        available_DBs = []
        for row in cur.execute("SHOW DATABASES").fetchall():
            available_DBs.append(row["name"])

        catalog = {}
        for db in available_DBs:
            catalog[db] = {}
            try:
                # SHOW OBJECTS includes views; SHOW TABLES would miss them
                objects = cur.execute(f'SHOW OBJECTS IN DATABASE "{db}"').fetchall()
            except snowflake.connector.errors.ProgrammingError as e:
                catalog[db] = {"error": e.msg}
                continue
            for t in objects:
                if t["schema_name"] != "INFORMATION_SCHEMA":
                    catalog[db].setdefault(t["schema_name"], []).append(t["name"])
        return catalog
    finally:
        conn.close()

In [6]:
import json

READ_ONLY_PREFIXES = ("SELECT", "WITH", "SHOW", "DESCRIBE", "DESC", "EXPLAIN")


@tool
def query(sql: str, limit: int = 100) -> dict:
    """Run one read-only SQL statement (SELECT, WITH, SHOW, DESCRIBE, EXPLAIN) on Snowflake.
    Use fully qualified names (DATABASE.SCHEMA.TABLE). Returns up to `limit` rows."""
    sql = sql.strip().rstrip(";").strip()
    if not sql.upper().startswith(READ_ONLY_PREFIXES):
        return {"error": "Only read-only queries (SELECT, WITH, SHOW, DESCRIBE, EXPLAIN) are allowed."}
    if ";" in sql:
        return {"error": "Only one statement per call is allowed."}

    # Push the limit into Snowflake so the warehouse doesn't compute rows we'd discard.
    # Fetch one extra row to detect truncation.
    if sql.upper().startswith(("SELECT", "WITH")):
        sql = f"SELECT * FROM (\n{sql}\n) LIMIT {limit + 1}"

    conn = get_snowflake_connection(get_snowflake_config())
    try:
        cur = conn.cursor(snowflake.connector.DictCursor)
        try:
            cur.execute(sql)
        except snowflake.connector.errors.ProgrammingError as e:
            return {"error": e.msg, "query_id": e.sfqid}

        rows = cur.fetchall()
        return {
            "query_id": cur.sfqid,
            "columns": [c.name for c in cur.description],
            "row_count": min(len(rows), limit),
            "truncated": len(rows) > limit,
            # round-trip through JSON so dates/decimals become plain strings for the LLM
            "rows": json.loads(json.dumps(rows[:limit], default=str)),
        }
    finally:
        conn.close()

In [8]:
from collections import Counter

import sqlglot
from sqlglot import exp

# Below this many micro-partitions a full scan is cheap, so pruning isn't worth flagging.
MIN_PARTITIONS_TO_FLAG = 10
PREDICATES = (exp.EQ, exp.NEQ, exp.GT, exp.GTE, exp.LT, exp.LTE, exp.Between, exp.In)


def _finding(severity: str, rule: str, detail: str, suggestion: str) -> dict:
    return {"severity": severity, "rule": rule, "detail": detail, "suggestion": suggestion}


def _static_findings(sql: str) -> list[dict]:
    """Anti-patterns visible from the SQL text alone."""
    try:
        tree = sqlglot.parse_one(sql, read="snowflake")
    except sqlglot.errors.ParseError as e:
        return [_finding("info", "parse_error", f"Static checks skipped: {e}", "")]

    findings, seen = [], set()

    def add(f):
        key = (f["rule"], f["detail"])
        if key not in seen:
            seen.add(key)
            findings.append(f)

    for select in tree.find_all(exp.Select):
        if any(e.is_star for e in select.expressions):
            add(_finding("medium", "select_star", "SELECT * reads every column.",
                         "Select only the columns you need; Snowflake is columnar, so unread columns are never scanned."))
        if select.args.get("distinct") and select.args.get("joins"):
            add(_finding("medium", "distinct_over_join", select.sql(dialect="snowflake")[:200],
                         "DISTINCT on a join often hides row fan-out. Fix the join keys or pre-aggregate before joining."))
        for e in select.expressions:
            if e.find(exp.Subquery):
                add(_finding("medium", "scalar_subquery_in_select", e.sql(dialect="snowflake")[:200],
                             "Rewrite as a JOIN to a pre-aggregated CTE, or as a window function."))
        if select.args.get("order") and select is not tree and not select.args.get("limit"):
            add(_finding("low", "order_by_in_subquery", select.args["order"].sql(dialect="snowflake"),
                         "ORDER BY inside a CTE/subquery without LIMIT is wasted sorting; order only the final result."))

    conditions = [w.this for w in tree.find_all(exp.Where)]
    for join in tree.find_all(exp.Join):
        on = join.args.get("on")
        if on is None:
            continue
        conditions.append(on)
        if on.find(exp.Or):
            add(_finding("high", "or_in_join", on.sql(dialect="snowflake"),
                         "OR in a join condition forces a cartesian join. Split into separate joins combined with UNION ALL."))

    for cond in conditions:
        for pred in cond.find_all(*PREDICATES):
            sides = [pred.this] if isinstance(pred, (exp.Between, exp.In)) else [pred.this, pred.expression]
            for side in sides:
                if isinstance(side, exp.Func) and side.find(exp.Column):
                    add(_finding("medium", "function_on_filter_column", pred.sql(dialect="snowflake"),
                                 "A function wrapped around a column can block partition pruning (Snowflake handles some, "
                                 "like YEAR on dates). Try comparing the raw column to a range, e.g. col >= '1995-01-01' AND "
                                 "col < '1996-01-01', and check with rewritten_sql whether partitions_assigned drops."))

    for like in tree.find_all(exp.Like, exp.ILike):
        pattern = like.expression
        if isinstance(pattern, exp.Literal) and pattern.is_string and pattern.this.startswith("%"):
            add(_finding("low", "leading_wildcard_like", like.sql(dialect="snowflake"),
                         "A leading % can't use pruning. Consider search optimization, or a prefix match if possible."))

    for union in tree.find_all(exp.Union):
        if union.args.get("distinct"):
            add(_finding("medium", "union_distinct", "UNION removes duplicates, which needs a full sort/hash.",
                         "Use UNION ALL if the branches can't produce duplicates."))

    # sqlglot's Snowflake dialect parses `x NOT IN (subquery)` as `x <> ALL (subquery)`
    not_in = [n for n in tree.find_all(exp.Not) if isinstance(n.this, exp.In) and n.this.args.get("query")]
    not_in += [n for n in tree.find_all(exp.NEQ) if isinstance(n.expression, exp.All)]
    for neg in not_in:
        add(_finding("medium", "not_in_subquery", neg.sql(dialect="snowflake")[:200],
                     "NOT IN (subquery) returns nothing if the subquery yields a NULL. Use NOT EXISTS instead."))

    for count in tree.find_all(exp.Count):
        if isinstance(count.this, exp.Distinct):
            add(_finding("low", "count_distinct", count.sql(dialect="snowflake"),
                         "If an approximate count is acceptable, APPROX_COUNT_DISTINCT is much cheaper."))

    return findings


def _table_metadata(cur, name: str) -> dict:
    db, schema, table = name.split(".")
    try:
        rows = cur.execute(f"SHOW TABLES LIKE '{table}' IN SCHEMA \"{db}\".\"{schema}\"").fetchall()
    except snowflake.connector.errors.ProgrammingError:
        return {}
    return {"rows": rows[0]["rows"], "cluster_by": rows[0]["cluster_by"] or None} if rows else {}


def _analyze(cur, sql: str) -> dict:
    """EXPLAIN the query (compiles only, no warehouse cost) and combine plan and static findings."""
    plan = json.loads(cur.execute(f"EXPLAIN USING JSON {sql}").fetchone()["content"])
    stats = plan.get("GlobalStats", {})
    ops = [op for step in plan.get("Operations", []) for op in step]

    total, assigned = stats.get("partitionsTotal", 0), stats.get("partitionsAssigned", 0)
    summary = {
        "partitions_total": total,
        "partitions_assigned": assigned,
        "pruning_pct": round(100 * (1 - assigned / total), 1) if total else None,
        "gb_scanned": round(stats.get("bytesAssigned", 0) / 1e9, 3),
        # bytesAssigned counts whole partitions, so column pruning (e.g. dropping SELECT *) only shows up here
        "columns_read": sum(len(op.get("expressions", [])) for op in ops if op["operation"] == "TableScan"),
        "operations": dict(Counter(op["operation"] for op in ops)),
    }

    findings, tables, metadata, flagged = [], [], {}, set()
    for op in ops:
        if op["operation"] == "CartesianJoin":
            findings.append(_finding("high", "cartesian_join", "; ".join(op.get("expressions", [])),
                                     "The plan multiplies every row by every row. Make sure each join has an equality "
                                     "condition and no OR."))
        if op["operation"] != "TableScan":
            continue
        name = op["objects"][0]
        if name not in metadata:
            metadata[name] = _table_metadata(cur, name)
        t_total, t_assigned = op.get("partitionsTotal", 0), op.get("partitionsAssigned", 0)
        tables.append({
            "table": name,
            "alias": op.get("alias"),
            "columns_read": len(op.get("expressions", [])),
            "partitions_assigned": t_assigned,
            "partitions_total": t_total,
            "gb_scanned": round(op.get("bytesAssigned", 0) / 1e9, 3),
            **metadata[name],
        })
        if t_total >= MIN_PARTITIONS_TO_FLAG and t_assigned / t_total >= 0.8 and name not in flagged:
            flagged.add(name)
            cluster_by = metadata[name].get("cluster_by")
            findings.append(_finding(
                "high" if op.get("bytesAssigned", 0) >= 1e9 else "medium",
                "poor_pruning",
                f"{name} scans {t_assigned}/{t_total} partitions.",
                f"Filter directly on the clustering key {cluster_by}." if cluster_by else
                "Add a selective filter on a column that tracks load order (usually a date). If this access pattern "
                "is frequent, a clustering key may help, but automatic clustering costs credits.",
            ))

    for name, n in Counter(t["table"] for t in tables).items():
        if n > 1:
            findings.append(_finding("medium", "repeated_scan", f"{name} is scanned {n} times.",
                                     "Read it once in a CTE, or use conditional aggregation (SUM(IFF(...))) instead of "
                                     "self-joins/unions."))

    findings += _static_findings(sql)
    order = {"high": 0, "medium": 1, "low": 2, "info": 3}
    findings.sort(key=lambda f: order[f["severity"]])
    return {"plan": summary, "tables": tables, "findings": findings}


def _pct_reduction(before: float, after: float) -> float | None:
    return round(100 * (before - after) / before, 1) if before else None


@tool
def optimize(sql: str, rewritten_sql: Optional[str] = None) -> dict:
    """Analyze a SELECT/WITH query for performance and cost problems, without running it.
    Returns the query plan (partitions and GB scanned per table) and findings ranked by severity,
    each with a suggested fix. Call it again with `rewritten_sql` to compare your rewrite's plan
    against the original before recommending it."""
    sql = sql.strip().rstrip(";").strip()
    if not sql.upper().startswith(("SELECT", "WITH")):
        return {"error": "optimize only accepts SELECT or WITH queries."}
    if ";" in sql:
        return {"error": "Only one statement per call is allowed."}

    conn = get_snowflake_connection(get_snowflake_config())
    try:
        cur = conn.cursor(snowflake.connector.DictCursor)
        try:
            result = _analyze(cur, sql)
        except snowflake.connector.errors.ProgrammingError as e:
            return {"error": f"Query does not compile: {e.msg}"}

        if rewritten_sql:
            rewritten_sql = rewritten_sql.strip().rstrip(";").strip()
            try:
                new = _analyze(cur, rewritten_sql)
            except snowflake.connector.errors.ProgrammingError as e:
                result["rewrite"] = {"error": f"Rewrite does not compile: {e.msg}"}
            else:
                before, after = result["plan"], new["plan"]
                result["rewrite"] = {
                    "plan": after,
                    "gb_scanned_reduction_pct": _pct_reduction(before["gb_scanned"], after["gb_scanned"]),
                    "partitions_reduction_pct": _pct_reduction(before["partitions_assigned"], after["partitions_assigned"]),
                    "columns_read_reduction_pct": _pct_reduction(before["columns_read"], after["columns_read"]),
                    "remaining_findings": new["findings"],
                    "note": "Compared plans only. Results were not checked; confirm both queries return the same rows.",
                }
        return result
    finally:
        conn.close()

In [9]:
import re

IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*$")
TEXT_TYPES = ("VARCHAR", "TEXT", "STRING", "CHAR")
DATE_TYPES = ("DATE", "TIMESTAMP")
SAMPLE_ROWS = 10_000  # sample values come from the first N rows only, to keep the scan small
MAX_TABLES = 10
MAX_VALUE_LEN = 60
HIGH_CARDINALITY = 1_000  # above this many distinct values (IDs, free text), samples don't help the LLM filter


def _quote(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def _describe_one(cur, table: str, sample_values: int) -> dict:
    parts = table.split(".")
    if len(parts) != 3 or not all(IDENTIFIER.match(p) for p in parts):
        return {"error": "Use a fully qualified name: DATABASE.SCHEMA.TABLE"}
    db, schema, name = (p.upper() for p in parts)
    fqn = f"{db}.{schema}.{name}"

    objects = cur.execute(f"SHOW OBJECTS LIKE '{name}' IN SCHEMA {db}.{schema}").fetchall()
    if not objects:
        return {"error": f"{table} not found or not authorized"}
    obj = objects[0]
    info = {
        "kind": obj["kind"],
        "rows": obj["rows"] if obj["kind"] == "TABLE" else None,  # Snowflake doesn't track view row counts
        "comment": obj["comment"] or None,
        "cluster_by": obj["cluster_by"] or None,
    }

    columns = [
        {
            "name": c["name"],
            "type": c["type"],
            "nullable": c["null?"] == "Y",
            **({"comment": c["comment"]} if c["comment"] else {}),
        }
        for c in cur.execute(f"DESC TABLE {fqn}").fetchall()
        if c["kind"] == "COLUMN"
    ]

    # Distinct example values for text columns, so the LLM writes WHERE country = 'India', not a guess.
    text_cols = [c for c in columns if c["type"].upper().startswith(TEXT_TYPES)]
    if text_cols and sample_values > 0:
        selects = ", ".join(
            f"ARRAY_SLICE(ARRAY_AGG(DISTINCT LEFT({_quote(c['name'])}, {MAX_VALUE_LEN})), 0, {sample_values}) "
            f"AS {_quote('v_' + c['name'])}, COUNT(DISTINCT {_quote(c['name'])}) AS {_quote('n_' + c['name'])}"
            for c in text_cols
        )
        cols = ", ".join(_quote(c["name"]) for c in text_cols)
        row = cur.execute(f"SELECT {selects} FROM (SELECT {cols} FROM {fqn} LIMIT {SAMPLE_ROWS})").fetchone()
        for c in text_cols:
            if row["n_" + c["name"]] > HIGH_CARDINALITY:
                c["high_cardinality"] = True
            else:
                c["sample_values"] = json.loads(row["v_" + c["name"]])
                c["distinct_in_sample"] = row["n_" + c["name"]]

    # Date ranges tell the LLM which periods exist. On tables MIN/MAX comes from metadata, so it's cheap;
    # on views it would run the whole view, so skip it there.
    date_cols = [c for c in columns if c["type"].upper().startswith(DATE_TYPES)]
    if date_cols and info["kind"] == "TABLE":
        selects = ", ".join(
            f"MIN({_quote(c['name'])}) AS {_quote('min_' + c['name'])}, "
            f"MAX({_quote(c['name'])}) AS {_quote('max_' + c['name'])}"
            for c in date_cols
        )
        row = cur.execute(f"SELECT {selects} FROM {fqn}").fetchone()
        for c in date_cols:
            c["min"], c["max"] = str(row["min_" + c["name"]]), str(row["max_" + c["name"]])

    return {**info, "columns": columns}


@tool
def describe_tables(tables: list[str], sample_values: int = 5) -> dict:
    """Get the columns of Snowflake tables or views before writing SQL against them.
    `tables` are fully qualified names (DATABASE.SCHEMA.TABLE), up to 10 per call.
    Returns each object's kind, row count and comment, plus every column's name, type and
    comment, example values for text columns, and min/max for date columns."""
    if len(tables) > MAX_TABLES:
        return {"error": f"At most {MAX_TABLES} tables per call."}

    conn = get_snowflake_connection(get_snowflake_config())
    try:
        cur = conn.cursor(snowflake.connector.DictCursor)
        result = {}
        for table in tables:
            try:
                result[table] = _describe_one(cur, table, sample_values)
            except snowflake.connector.errors.ProgrammingError as e:
                result[table] = {"error": e.msg}
        return result
    finally:
        conn.close()

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeout
from functools import lru_cache
from typing import TypedDict

import groq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from pydantic import Field

# Models available on this Groq account (checked with client.models.list()). Override in .env.
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
GROQ_FALLBACK_MODEL = os.getenv("GROQ_FALLBACK_MODEL", "openai/gpt-oss-20b")  # used if the primary errors/rate-limits
MAX_SQL_ATTEMPTS = 3          # generate -> validate/execute -> fix loops before giving up
STRUCTURED_OUTPUT_RETRIES = 2  # Groq sometimes returns malformed tool calls; retry those (not API errors)
LLM_TIMEOUT_S = 30             # per HTTP read; Groq's client doesn't bound total time, so see LLM_DEADLINE_S
LLM_DEADLINE_S = 90            # hard wall-clock limit per LLM call, including retries and the fallback model
RESULT_LIMIT = 100
MIN_GB_TO_OPTIMIZE = 0.1     # below this, a rewrite can't save meaningful credits; skip the extra LLM call
EXCLUDED_DATABASES = {"SNOWFLAKE"}  # system database; hundreds of objects that would flood the prompt


class AgentError(RuntimeError):
    """A failure the agent can't recover from by rewriting SQL (LLM down, Snowflake unreachable, ...)."""


@lru_cache(maxsize=1)
def get_llms() -> list[ChatGroq]:
    if not os.getenv("GROQ_API_KEY"):
        raise AgentError("GROQ_API_KEY is not set. Add it to .env and restart the kernel.")
    models = [GROQ_MODEL] + ([GROQ_FALLBACK_MODEL] if GROQ_FALLBACK_MODEL and GROQ_FALLBACK_MODEL != GROQ_MODEL else [])
    # max_retries covers rate limits (429) and transient 5xx errors with backoff, per model
    return [ChatGroq(model=m, temperature=0, max_retries=1, timeout=LLM_TIMEOUT_S) for m in models]


def _with_fallbacks(runnables: list):
    primary, *rest = runnables
    return primary.with_fallbacks(rest) if rest else primary


_llm_pool = ThreadPoolExecutor(max_workers=4, thread_name_prefix="llm")


def _invoke_with_deadline(runnable, messages):
    """The HTTP timeout only limits each read, and a slow/queued Groq response once held a call for 16 minutes.
    Enforce a total deadline from outside. On timeout the worker thread is abandoned (it finishes in the background)."""
    future = _llm_pool.submit(runnable.invoke, messages)
    try:
        return future.result(timeout=LLM_DEADLINE_S)
    except FutureTimeout:
        future.cancel()
        raise AgentError(f"LLM did not respond within {LLM_DEADLINE_S}s (Groq may be overloaded or rate-limiting).")


def call_llm(schema: type[BaseModel], system: str, user: str) -> BaseModel:
    """Ask the LLM for structured output, falling back to the second model and retrying malformed responses."""
    llm = _with_fallbacks([m.with_structured_output(schema) for m in get_llms()])
    last_exc = None
    for _ in range(STRUCTURED_OUTPUT_RETRIES):
        try:
            out = _invoke_with_deadline(llm, [SystemMessage(content=system), HumanMessage(content=user)])
            if out is not None:
                return out
            last_exc = ValueError("empty structured output")
        except AgentError:
            raise
        except groq.APIError as e:
            # Timeouts, rate limits, outages: both models were already tried, so retrying again only adds wait time.
            raise AgentError(f"LLM unavailable: {type(e).__name__}: {e}")
        except Exception as e:  # malformed tool call / output that failed schema validation
            last_exc = e
    raise AgentError(f"LLM call failed: {type(last_exc).__name__}: {last_exc}")


def call_llm_text(system: str, user: str) -> str:
    try:
        return _invoke_with_deadline(
            _with_fallbacks(get_llms()), [SystemMessage(content=system), HumanMessage(content=user)]
        ).content
    except AgentError:
        raise
    except Exception as e:
        raise AgentError(f"LLM call failed: {type(e).__name__}: {e}")


class TableSelection(BaseModel):
    reasoning: str = Field(description="Why these tables answer the question, or why none do")
    answerable: bool = Field(description="False if no listed table actually contains the data the question asks about")
    tables: list[str] = Field(description="Fully qualified DATABASE.SCHEMA.TABLE names, copied exactly from the list; "
                                          "empty if not answerable")


class GeneratedSQL(BaseModel):
    sql: str = Field(description="One Snowflake SELECT or WITH statement, no trailing semicolon")
    explanation: str = Field(description="One sentence on how the query answers the question")


class RewrittenSQL(BaseModel):
    sql: str = Field(description="The optimized query; must return exactly the same rows as the original")
    changes: str = Field(description="What was changed and why")


class AgentState(TypedDict, total=False):
    question: str
    tables: list[str]
    schema: str
    sql: str
    explanation: str
    attempts: int
    last_error: Optional[str]     # recoverable: SQL failed to compile/run, fed back to generate_sql
    error: Optional[str]          # fatal: routes to the fail node
    plan: dict
    findings: list[dict]
    optimization: dict
    result: dict
    answer: str


def format_schema(described: dict) -> str:
    """Compact text version of describe_tables output, to save prompt tokens."""
    lines = []
    for table, d in described.items():
        meta = d["kind"] + (f", {d['rows']:,} rows" if d.get("rows") else "")
        lines.append(f"{table} ({meta})" + (f" -- {d['comment']}" if d.get("comment") else ""))
        for c in d["columns"]:
            notes = []
            if c.get("comment"):
                notes.append(c["comment"])
            if "sample_values" in c:
                notes.append("values: " + ", ".join(repr(v) for v in c["sample_values"]))
            if "min" in c:
                notes.append(f"range {c['min']} .. {c['max']}")
            if c.get("high_cardinality"):
                notes.append("high cardinality")
            col_type = c["type"].replace("(16777216)", "")
            lines.append(f"  {c['name']} {col_type}" + (f"  -- {'; '.join(notes)}" if notes else ""))
    return "\n".join(lines)

In [ ]:
import functools
import logging

logger = logging.getLogger("snowflake_agent")

SQL_RULES = """Rules:
- Write ONE Snowflake SQL SELECT (or WITH ... SELECT) statement. No DDL/DML, no trailing semicolon.
- Use only the tables and columns listed in the schema. Always use fully qualified names (DATABASE.SCHEMA.TABLE).
- When filtering text columns, use the exact spelling and case shown in `values:`; use ILIKE if unsure.
- Select only the columns needed to answer (never SELECT *).
- Filter on raw columns (e.g. date_col >= '2020-01-01'), not functions of columns, so Snowflake can prune.
- For "top N" questions use ORDER BY ... LIMIT N. Aggregate in SQL rather than returning raw rows."""


def guarded(fn):
    """Turn any unexpected exception inside a node into a fatal error instead of crashing the graph."""
    @functools.wraps(fn)
    def wrapper(state: AgentState) -> dict:
        try:
            return fn(state)
        except AgentError as e:
            return {"error": str(e)}
        except snowflake.connector.errors.Error as e:  # connection/auth/network problems
            return {"error": f"Snowflake error in {fn.__name__}: {e}"}
        except Exception as e:
            logger.exception("node %s failed", fn.__name__)
            return {"error": f"Unexpected error in {fn.__name__}: {type(e).__name__}: {e}"}
    return wrapper


@guarded
def select_tables(state: AgentState) -> dict:
    catalog = retrieval.invoke({})
    available = [
        f"{db}.{schema}.{table}"
        for db, schemas in catalog.items() if db not in EXCLUDED_DATABASES
        for schema, tables in schemas.items() if isinstance(tables, list)
        for table in tables
    ]
    if not available:
        return {"error": "No tables are visible to this Snowflake role."}

    selection = call_llm(
        TableSelection,
        "You pick the Snowflake tables needed to answer a question. Choose the fewest tables that "
        "fully answer it (include dimension tables needed for joins/labels). Prefer MART over STAGING/RAW. "
        "If no table actually contains the subject of the question (e.g. the question is about car sales and "
        "the tables hold health or retail benchmark data), set answerable=false. Never stretch unrelated data "
        "to fit the question.",
        f"Question: {state['question']}\n\nAvailable tables:\n" + "\n".join(available),
    )
    if not selection.answerable:
        return {"error": f"The available data doesn't cover this question. {selection.reasoning}"}
    lookup = {t.upper(): t for t in available}
    tables = [lookup[t.strip().upper()] for t in selection.tables if t.strip().upper() in lookup]
    if not tables:
        return {"error": f"Couldn't find tables relevant to the question. LLM suggested: {selection.tables}"}
    return {"tables": tables[:MAX_TABLES]}


@guarded
def describe(state: AgentState) -> dict:
    described = describe_tables.invoke({"tables": state["tables"]})
    ok = {t: d for t, d in described.items() if "error" not in d}
    if not ok:
        errors = {t: d["error"] for t, d in described.items()}
        return {"error": f"Couldn't describe any of the selected tables: {errors}"}
    return {"schema": format_schema(ok), "tables": list(ok)}


@guarded
def generate_sql(state: AgentState) -> dict:
    prompt = f"Schema:\n{state['schema']}\n\nQuestion: {state['question']}"
    if state.get("last_error"):
        prompt += (
            f"\n\nYour previous query failed. Fix it.\nPrevious query:\n{state['sql']}\n"
            f"Error:\n{state['last_error']}"
        )
    out = call_llm(GeneratedSQL, "You write Snowflake SQL to answer questions.\n" + SQL_RULES, prompt)
    return {
        "sql": out.sql.strip().rstrip(";").strip(),
        "explanation": out.explanation,
        "attempts": state.get("attempts", 0) + 1,
        "last_error": None,
    }


@guarded
def validate(state: AgentState) -> dict:
    """Compile with EXPLAIN (no warehouse cost). Compile errors go back to generate_sql."""
    report = optimize.invoke({"sql": state["sql"]})
    if "error" in report:
        return {"last_error": report["error"]}
    return {"plan": report["plan"], "findings": report["findings"]}


@guarded
def optimize_sql(state: AgentState) -> dict:
    """Try one LLM rewrite for high/medium findings. Never fatal: on any problem, keep the original SQL."""
    actionable = [f for f in state.get("findings", []) if f["severity"] in ("high", "medium")]
    if not actionable:
        return {"optimization": {"status": "skipped", "reason": "no high/medium findings"}}
    has_high = any(f["severity"] == "high" for f in actionable)
    if not has_high and state["plan"]["gb_scanned"] < MIN_GB_TO_OPTIMIZE:
        return {"optimization": {"status": "skipped",
                                 "reason": f"scans only {state['plan']['gb_scanned']} GB and no high findings"}}

    try:
        issues = "\n".join(f"- [{f['severity']}] {f['rule']}: {f['detail']} -> {f['suggestion']}" for f in actionable)
        out = call_llm(
            RewrittenSQL,
            "You optimize Snowflake SQL for cost. The rewrite MUST return exactly the same rows and columns. "
            "If a finding can't be fixed without changing results, leave that part unchanged.\n" + SQL_RULES,
            f"Schema:\n{state['schema']}\n\nQuery:\n{state['sql']}\n\nFindings:\n{issues}",
        )
        rewritten = out.sql.strip().rstrip(";").strip()
        report = optimize.invoke({"sql": state["sql"], "rewritten_sql": rewritten})
    except (AgentError, snowflake.connector.errors.Error) as e:
        return {"optimization": {"status": "failed", "reason": str(e)}}

    rewrite = report.get("rewrite", {})
    if "error" in report or "error" in rewrite:
        return {"optimization": {"status": "rejected", "reason": report.get("error") or rewrite["error"]}}

    reduction = rewrite["gb_scanned_reduction_pct"] or 0
    columns_reduction = rewrite["columns_read_reduction_pct"] or 0
    high_before = sum(f["severity"] == "high" for f in state["findings"])
    high_after = sum(f["severity"] == "high" for f in rewrite["remaining_findings"])
    if reduction > 0 or (reduction == 0 and (columns_reduction > 0 or high_after < high_before)):
        return {
            "sql": rewritten,
            "plan": rewrite["plan"],
            "findings": rewrite["remaining_findings"],
            "optimization": {"status": "applied", "changes": out.changes, "original_sql": state["sql"],
                             "gb_scanned_reduction_pct": reduction, "columns_read_reduction_pct": columns_reduction},
        }
    return {"optimization": {"status": "rejected",
                             "reason": f"no improvement (GB scanned {reduction}%, columns read {columns_reduction}%)"}}


@guarded
def execute(state: AgentState) -> dict:
    result = query.invoke({"sql": state["sql"], "limit": RESULT_LIMIT})
    if "error" in result:  # runtime errors, e.g. division by zero or bad casts
        return {"last_error": result["error"]}
    return {"result": result}


@guarded
def write_answer(state: AgentState) -> dict:
    result = state["result"]
    if result["row_count"] == 0:
        return {"answer": "The query ran successfully but returned no rows."}
    rows = json.dumps(result["rows"][:50], default=str)[:8000]  # cap tokens for Groq rate limits
    try:
        text = call_llm_text(
            "Answer the user's question in plain language using only the query result. Be concise. "
            "If the result is truncated, say the answer covers only the first rows.",
            f"Question: {state['question']}\nSQL:\n{state['sql']}\n"
            f"Truncated: {result['truncated']}\nRows ({result['row_count']}):\n{rows}",
        )
    except AgentError as e:
        # The data is still useful even if summarizing failed.
        text = f"(Couldn't summarize: {e}) Raw result: {rows[:2000]}"
    return {"answer": text}


def fail(state: AgentState) -> dict:
    if state.get("error"):
        reason = state["error"]
    else:
        reason = f"SQL still failing after {state.get('attempts', 0)} attempts. Last error: {state.get('last_error')}"
    return {"answer": f"Sorry, I couldn't answer that. {reason}"}

In [ ]:
from langgraph.errors import GraphRecursionError


def next_or_fail(next_node: str):
    def route(state: AgentState) -> str:
        return "fail" if state.get("error") else next_node
    return route


def retry_or(next_node: str):
    """After validate/execute: fatal -> fail, SQL error -> regenerate (until attempts run out), else continue."""
    def route(state: AgentState) -> str:
        if state.get("error"):
            return "fail"
        if state.get("last_error"):
            return "generate_sql" if state.get("attempts", 0) < MAX_SQL_ATTEMPTS else "fail"
        return next_node
    return route


builder = StateGraph(AgentState)
for name, node in [
    ("select_tables", select_tables), ("describe", describe), ("generate_sql", generate_sql),
    ("validate", validate), ("optimize_sql", optimize_sql), ("execute", execute),
    ("write_answer", write_answer), ("fail", fail),
]:
    builder.add_node(name, node)

builder.add_edge(START, "select_tables")
builder.add_conditional_edges("select_tables", next_or_fail("describe"), ["describe", "fail"])
builder.add_conditional_edges("describe", next_or_fail("generate_sql"), ["generate_sql", "fail"])
builder.add_conditional_edges("generate_sql", next_or_fail("validate"), ["validate", "fail"])
builder.add_conditional_edges("validate", retry_or("optimize_sql"), ["optimize_sql", "generate_sql", "fail"])
builder.add_conditional_edges("optimize_sql", next_or_fail("execute"), ["execute", "fail"])
builder.add_conditional_edges("execute", retry_or("write_answer"), ["write_answer", "generate_sql", "fail"])
builder.add_edge("write_answer", END)
builder.add_edge("fail", END)

graph = builder.compile()


def ask(question: str) -> AgentState:
    try:
        return graph.invoke({"question": question, "attempts": 0}, {"recursion_limit": 30})
    except GraphRecursionError:
        return {"question": question, "answer": "Sorry, the agent got stuck in a loop and was stopped."}

In [ ]:
state = ask("Which 5 countries had the highest life expectancy for both sexes in the most recent year?")

print(state["answer"])
print("\nSQL:\n", state.get("sql"))
print("\nAttempts:", state.get("attempts"), "| Optimization:", state.get("optimization"))